# 📊 Telco Customer Churn Prediction
An end-to-end machine learning project predicting whether telecom customers will churn, using Scikit-Learn Pipelines, ColumnTransformer, and Hyperparameter Tuning with GridSearchCV.

- **Dataset:** Telco Customer Churn
- **Target Variable:** `Churn` (1 = Customer Left, 0 = Customer Stayed)


## 1. Import Libraries


In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Load and Inspect the Dataset


In [2]:
df = pd.read_csv("Telco-Customer-Churn.csv")

print("Dataset Shape:", df.shape)
df.head()


Dataset Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## 3. Data Cleaning and Preparation
1. Drop `customerID` (unique identifier that carries no predictive signal).
2. Convert `TotalCharges` to numeric (empty space strings are converted to NaN).
3. Map target variable `Churn` to binary integers (`0` = No, `1` = Yes).


In [4]:
# Drop customerID
df.drop("customerID", axis=1, inplace=True)


In [5]:
# Convert TotalCharges to float; blank spaces are coerced to NaN
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)


In [6]:
# Map target variable Churn (No -> 0, Yes -> 1)
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("Target Class Distribution:")
print(df["Churn"].value_counts())


Target Class Distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


## 4. Train-Test Split
Split features `X` and target `y` using stratification to preserve the churn ratio in both training (80%) and testing (20%) sets.


In [7]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")


Training samples: 5634
Testing samples:  1409


## 5. Build Scikit-Learn Preprocessing Pipelines
- **Numeric Features:** Impute missing values with `median`, then scale with `StandardScaler`.
- **Categorical Features:** Impute missing values with `most_frequent`, then encode with `OneHotEncoder(handle_unknown='ignore')`.
- Combine both pipelines into a single `ColumnTransformer`.


In [8]:
# Identify numeric and categorical columns
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X.select_dtypes(
    include=["object"]
).columns

print("Numeric Features:", list(numeric_features))
print("Categorical Features:", list(categorical_features))


Numeric Features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical Features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


/var/folders/9r/3ycx2rcd15q82v7bghd24bx00000gn/T/ipykernel_19676/214045540.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [9]:
# Preprocessing pipeline for numeric data
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocessing pipeline for categorical data
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine transformers using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


## 6. Model 1: Logistic Regression with Hyperparameter Tuning


In [10]:
# Pipeline combining preprocessor and Logistic Regression
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

# Hyperparameter search space
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10]
}

# 5-fold cross-validation with GridSearchCV
grid_lr = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_lr.fit(X_train, y_train)

print("Best Parameters:", grid_lr.best_params_)
print(f"Best CV Accuracy: {grid_lr.best_score_:.4f}")


Best Parameters: {'classifier__C': 10}
Best CV Accuracy: 0.8046


In [11]:
# Evaluate Logistic Regression on test set
lr_prediction = grid_lr.predict(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, lr_prediction):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, lr_prediction, target_names=["No Churn (0)", "Churn (1)"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, lr_prediction))


Test Accuracy: 0.8055

Classification Report:
              precision    recall  f1-score   support

No Churn (0)       0.85      0.89      0.87      1035
   Churn (1)       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

Confusion Matrix:
[[926 109]
 [165 209]]


## 7. Model 2: Random Forest Classifier with Hyperparameter Tuning


In [12]:
# Pipeline combining preprocessor and Random Forest
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

rf_params = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [5, 10, None],
    "classifier__min_samples_split": [2, 5]
}

grid_rf = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print("Best Parameters:", grid_rf.best_params_)
print(f"Best CV Accuracy: {grid_rf.best_score_:.4f}")


Best Parameters: {'classifier__max_depth': 10, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
Best CV Accuracy: 0.8010


In [13]:
# Evaluate Random Forest on test set
rf_prediction = grid_rf.predict(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, rf_prediction):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, rf_prediction, target_names=["No Churn (0)", "Churn (1)"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_prediction))


Test Accuracy: 0.7963

Classification Report:
              precision    recall  f1-score   support

No Churn (0)       0.84      0.89      0.87      1035
   Churn (1)       0.64      0.53      0.58       374

    accuracy                           0.80      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409

Confusion Matrix:
[[925 110]
 [177 197]]


## 8. Export the Best Performing Model Pipeline
Logistic Regression achieved slightly higher Accuracy and F1-Score. We export the trained `best_estimator_` using `joblib` so it is ready for deployment.


In [14]:
# Save the best estimator pipeline (preprocessor + classifier)
joblib.dump(
    grid_lr.best_estimator_,
    "logistic_pipeline.pkl"
)

print("Best model pipeline saved successfully to 'logistic_pipeline.pkl'!")


Best model pipeline saved successfully to 'logistic_pipeline.pkl'!


## 9. Make Predictions on New Customer Data
Load the saved pipeline and pass raw customer data directly — the pipeline automatically handles all scaling and encoding!


In [15]:
# Load the saved pipeline
loaded_pipeline = joblib.load("logistic_pipeline.pkl")

# New customer profile
new_customer = pd.DataFrame({
    "gender": ["Male"],
    "SeniorCitizen": [0],
    "Partner": ["Yes"],
    "Dependents": ["No"],
    "tenure": [12],
    "PhoneService": ["Yes"],
    "MultipleLines": ["No"],
    "InternetService": ["Fiber optic"],
    "OnlineSecurity": ["No"],
    "OnlineBackup": ["Yes"],
    "DeviceProtection": ["No"],
    "TechSupport": ["No"],
    "StreamingTV": ["Yes"],
    "StreamingMovies": ["Yes"],
    "Contract": ["Month-to-month"],
    "PaperlessBilling": ["Yes"],
    "PaymentMethod": ["Electronic check"],
    "MonthlyCharges": [75.50],
    "TotalCharges": [906.00]
})

# Predict
prediction = loaded_pipeline.predict(new_customer)[0]
probability = loaded_pipeline.predict_proba(new_customer)[0][1]

status = "Will Churn" if prediction == 1 else "Will Stay (No Churn)"
print(f"Prediction: {status} (Class {prediction})")
print(f"Churn Probability: {probability:.2%}")


Prediction: Will Churn (Class 1)
Churn Probability: 88.84%
